# EDA — ciclos que começam no goleiro

Analisa se as features defensivas agregadas por ciclo de posse se relacionam com o pico de ameaça (`threat_score`), só para ciclos que começam com o goleiro e os 11 defensores atrás da bola. Também gera `data/relevant_features.json`, usado pelo modelo (`4_model`).

## 1. Setup

Imports, sessão Spark e funções reutilizáveis (existem cópias em `src/utils/`, ainda não importadas daqui).

### 1.1 Imports e Spark

In [1]:
import pandas as pd
from pathlib import Path
import os
import sys

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

import numpy as np
from scipy import stats

import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option('display.max_columns', None)

In [2]:
spark = (
    SparkSession
    .builder
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.python.worker.faulthandler.enabled", "true")  # traceback nativo se um worker crashar
    .master("local[*]")
    .appName("eda_cycle_start_gk")
    .getOrCreate()
)

### 1.2 Agregação de features e target

`build_feature_target_stats_df`: uma linha por ciclo, com todas as agregações.

In [3]:
AGG_FUNCS_FEATURES = {
    "avg": F.mean,
    "median": F.median,
    #"var": F.variance,
    #"std": F.stddev,
    "min": F.min,
    #"p5":  lambda c: F.percentile_approx(c, 0.05),
    "max": F.max,
    #"p95": lambda c: F.percentile_approx(c, 0.95),
}

AGG_FUNCS_TARGET = {
    "avg": F.mean,
    "sum": F.sum,
    "max": F.max,
}


def build_feature_target_stats_df(df, features, target, group_cols):
    """Agrega features e target por `group_cols` num único groupBy (colunas "{agg}_{coluna}"). Assume que eventos com tracking inconsistente já foram removidos do df."""
    if isinstance(features, str):
        features = [features]

    agg_exprs = []

    for col_name in features:
        for prefix, func in AGG_FUNCS_FEATURES.items():
            alias = f"{prefix}_{col_name}"
            agg_exprs.append(F.round(func(F.col(col_name)), 3).alias(alias))

    for prefix, func in AGG_FUNCS_TARGET.items():
        alias = f"{prefix}_{target}"
        agg_exprs.append(F.round(func(F.col(target)), 3).alias(alias))

    df_agg = df.groupBy(*group_cols).agg(*agg_exprs)

    return df_agg

### 1.3 Heatmaps de correlação

Matriz de correlação (Spearman) com `*` para p < 0.05: geral e por time.

In [4]:
def _corr_pvalue_matrix(df_pd, cols, method='spearman'):
    """Matriz de correlação e de p-valor par-a-par (pares completos)."""
    corr_func = stats.spearmanr if method == 'spearman' else stats.pearsonr

    corr = df_pd[cols].corr(method=method)

    pvals = pd.DataFrame(np.nan, index=cols, columns=cols)
    for i, col_i in enumerate(cols):
        for j, col_j in enumerate(cols):
            if j < i:
                continue
            if i == j:
                # correlação de uma coluna com ela mesma é trivial (r=1, p=0)
                pvals.loc[col_i, col_j] = 0.0
                continue
            paired = df_pd[[col_i, col_j]].dropna()
            pvalue = np.nan if len(paired) < 3 else corr_func(paired[col_i], paired[col_j])[1]
            pvals.loc[col_i, col_j] = pvalue
            pvals.loc[col_j, col_i] = pvalue

    return corr, pvals


def _corr_text_with_significance(corr, pvals, alpha):
    """Texto da célula: correlação arredondada, com "*" quando p < alpha."""
    text = corr.round(3).astype(str)
    return text.where(pvals >= alpha, text + '*')

In [5]:
def plot_correlation_heatmap(df_agg, features, target, target_aggs=None, title_suffix="", alpha=0.05):
    """UM heatmap (Spearman): todas as agregações das features x agregação(ões) do target. "*" quando p < alpha."""
    if isinstance(features, str):
        features = [features]
    if target_aggs is None:
        target_aggs = list(AGG_FUNCS_TARGET.keys())
    elif isinstance(target_aggs, str):
        target_aggs = [target_aggs]

    df_agg_pd = df_agg.toPandas() if not isinstance(df_agg, pd.DataFrame) else df_agg

    agg_names = list(AGG_FUNCS_FEATURES.keys())
    feature_cols = [f"{prefix}_{feature}" for feature in features for prefix in agg_names]
    target_cols = [f"{ta}_{target}" for ta in target_aggs]

    corr, pvals = _corr_pvalue_matrix(df_agg_pd, feature_cols + target_cols)
    text = _corr_text_with_significance(corr, pvals, alpha)

    fig = go.Figure(
        data=go.Heatmap(
            z=corr.values,
            x=corr.columns,
            y=corr.columns,
            colorscale='RdBu',
            zmin=-1,
            zmax=1,
            text=text.values,
            texttemplate="%{text}",
            colorbar=dict(title="Correlação"),
        )
    )

    fig.update_layout(
        title=f"Correlação (Spearman) — agregações das features x {'/'.join(target_aggs)}_{target}{title_suffix}",
        template="simple_white",
        height=700,
        width=1000,
        xaxis=dict(tickangle=-45),
    )

    fig.show()

In [6]:
def plot_correlation_heatmap_by_team(df_agg, features, target, team_col, feature_agg, target_aggs='max', max_teams=5, alpha=0.05):
    """Um bloco de heatmap(s) por time (até `max_teams`). `feature_agg`: str (mesma agregação pra todas) ou dict `AGG_FEATURES` (um subplot por posição no ranking). "*" quando p < alpha."""
    if isinstance(features, str):
        features = [features]
    if isinstance(target_aggs, str):
        target_aggs = [target_aggs]

    df_agg_pd = df_agg.toPandas() if not isinstance(df_agg, pd.DataFrame) else df_agg

    target_cols = [f"{ta}_{target}" for ta in target_aggs]
    rank_maps = _resolve_feature_agg_ranks(features, feature_agg)

    teams = sorted(df_agg_pd[team_col].dropna().unique())[:max_teams]

    for team in teams:
        df_team_pd = df_agg_pd[df_agg_pd[team_col] == team]

        matrices = []
        for rank_map in rank_maps:
            feature_cols = [f"{rank_map[feature]}_{feature}" for feature in features if rank_map[feature] is not None]
            corr, pvals = _corr_pvalue_matrix(df_team_pd, feature_cols + target_cols)
            text = _corr_text_with_significance(corr, pvals, alpha)
            matrices.append((corr.values, list(corr.columns), list(corr.columns), text.values))

        row_titles = [f"Agregação #{i + 1}" for i in range(len(rank_maps))] if len(rank_maps) > 1 else [""]
        fig_title = f"Correlação (Spearman) x {'/'.join(target_aggs)}_{target} — {team}"

        _plot_heatmap_grid(matrices, row_titles, fig_title)

### 1.4 Heatmaps por faixa de posição

Correlação feature x target calculada dentro de cada faixa do campo. Aceita uma agregação única ou o dict `AGG_FEATURES`.

In [7]:
def _resolve_feature_agg_ranks(features, feature_agg):
    """Converte `feature_agg` (str ou dict AGG_FEATURES) numa lista de rank maps {feature: agregação}; rank 0 = melhor agregação de cada feature."""
    if isinstance(feature_agg, str):
        return [{feature: feature_agg for feature in features}]

    max_ranks = max((len(feature_agg.get(feature, [])) for feature in features), default=0)
    rank_maps = []
    for rank_idx in range(max_ranks):
        rank_map = {}
        for feature in features:
            entries = feature_agg.get(feature, [])
            if rank_idx < len(entries):
                rank_map[feature] = next(key for key in entries[rank_idx] if key != 'p-value')
            else:
                rank_map[feature] = None
        rank_maps.append(rank_map)
    return rank_maps


def _plot_heatmap_grid(matrices, row_titles, fig_title, width=1100):
    """Empilha um heatmap por item de `matrices` (z, x_labels, y_labels, text), com escala de cor compartilhada."""
    n_rows = len(matrices)
    if n_rows == 0:
        return

    fig = make_subplots(
        rows=n_rows,
        cols=1,
        subplot_titles=row_titles,
        vertical_spacing=min(0.15, 0.6 / n_rows) if n_rows > 1 else 0.1,
    )

    row_heights = []
    for i, (z, x_labels, y_labels, text) in enumerate(matrices, start=1):
        fig.add_trace(
            go.Heatmap(
                z=z,
                x=x_labels,
                y=y_labels,
                coloraxis="coloraxis",
                text=text,
                texttemplate="%{text}",
            ),
            row=i,
            col=1,
        )
        fig.update_xaxes(tickangle=-45, row=i, col=1)
        row_heights.append(max(350, 28 * len(y_labels) + 100))

    fig.update_layout(
        title=fig_title,
        template="simple_white",
        height=sum(row_heights),
        width=width,
        coloraxis=dict(colorscale='RdBu', cmin=-1, cmax=1, colorbar=dict(title="Correlação")),
    )

    fig.show()


def _build_position_bin_matrix(df_agg_pd, features, rank_map, target, target_agg, bin_col, alpha):
    """Matriz features x faixas (correlação Spearman dentro de cada faixa) para um rank map."""
    target_col = f"{target_agg}_{target}"
    bin_values = [b for b in df_agg_pd[bin_col].cat.categories if b in df_agg_pd[bin_col].values]

    z = []
    p = []
    for feature in features:
        prefix = rank_map.get(feature)
        row_z = []
        row_p = []
        for b in bin_values:
            if prefix is None:
                row_z.append(np.nan)
                row_p.append(np.nan)
                continue
            feature_col = f"{prefix}_{feature}"
            df_bin = df_agg_pd[df_agg_pd[bin_col] == b][[feature_col, target_col]].dropna()
            if len(df_bin) >= 3:
                r, pvalue = stats.spearmanr(df_bin[feature_col], df_bin[target_col])
            else:
                r, pvalue = np.nan, np.nan
            row_z.append(r)
            row_p.append(pvalue)
        z.append(row_z)
        p.append(row_p)

    x_labels = [str(b) for b in bin_values]
    z = np.array(z)
    p = np.array(p)
    text = np.where(p < alpha, np.char.add(np.round(z, 3).astype(str), '*'), np.round(z, 3).astype(str))

    # rótulo de cada feature no eixo y já mostra a agregação usada nesse rank
    y_labels = [f"{feature} ({rank_map.get(feature) or 'N/A'})" for feature in features]

    return z, x_labels, y_labels, text


def _plot_position_bin_heatmap(df_agg_pd, features, target, feature_agg, target_agg, bin_col, title_suffix="", alpha=0.05):
    """Desenha o(s) heatmap(s) features x faixas: str -> um heatmap; dict AGG_FEATURES -> um subplot por ranking."""
    rank_maps = _resolve_feature_agg_ranks(features, feature_agg)
    matrices = [
        _build_position_bin_matrix(df_agg_pd, features, rank_map, target, target_agg, bin_col, alpha)
        for rank_map in rank_maps
    ]

    bin_labels = matrices[0][1] if matrices else []
    row_titles = [f"Agregação #{i + 1}" for i in range(len(rank_maps))] if len(rank_maps) > 1 else [""]
    fig_title = f"Correlação (Spearman) por faixa de {bin_col} x {target_agg}_{target}{title_suffix}<br><sup>Faixas: {' | '.join(bin_labels)}</sup>"

    _plot_heatmap_grid(matrices, row_titles, fig_title)


def plot_correlation_by_position_bin(df_agg_pd, features, target, feature_agg, target_agg, bin_col, alpha=0.05):
    """Heatmap features x faixas de `bin_col`: correlação com o target calculada dentro de cada faixa (checa confundimento posicional). "*" quando p < alpha."""
    if isinstance(features, str):
        features = [features]

    _plot_position_bin_heatmap(df_agg_pd, features, target, feature_agg, target_agg, bin_col, alpha=alpha)


def plot_correlation_by_position_bin_by_team(df_agg_pd, features, target, team_col, feature_agg, target_agg, bin_col, max_teams=5, alpha=0.05):
    """Igual a `plot_correlation_by_position_bin`, um bloco por time (até `max_teams`)."""
    if isinstance(features, str):
        features = [features]

    teams = sorted(df_agg_pd[team_col].dropna().unique())[:max_teams]

    for team in teams:
        df_team_pd = df_agg_pd[df_agg_pd[team_col] == team]
        _plot_position_bin_heatmap(
            df_team_pd, features, target, feature_agg, target_agg, bin_col, title_suffix=f" — {team}", alpha=alpha
        )

## 2. Dados

### 2.1 Leitura

Base de ameaça (parquet) e base de features (csv).

In [8]:
# caminho pra pasta com dados
data_folder_path = Path().resolve().parent.parent / "data"

In [9]:
# base de ameaça criada
threat_dataset_path = str(data_folder_path / "threat_dataset")
features_dataset_path = str(data_folder_path / "features_dataset")

df_threat = spark.read.parquet(threat_dataset_path)
df_features = spark.read.csv(features_dataset_path, header=True, inferSchema=True)

### 2.2 Base por evento

Junta ameaça e features por evento, remove os eventos com tracking inconsistente (`is_tracking_inconsistent`) e cria colunas auxiliares: `first_cycle_event`, `defendingTeamName` (time que defende) e `ball_x` (posição x da bola).

In [10]:
threat_cols = [
    'gameId',
    'competitionId',
    'season',
    'date',
    'eventId',
    'period',
    'startGameClock',
    'startFormattedGameClock',
    'homeTeam',
    'flipped_homeTeam',
    'eventType',
    'eventTypeDescription',
    'eventSubTypeDescription',
    'eventOutcomeDescription',
    'eventPlayerName',
    'eventPlayerPositionType',
    'eventPlayerPositionGroup',
    'eventTeamName',
    'competitionName',
    'homeTeamName',
    'opponentTeamName',
    'possession_id',
    'attackers',
    'defenders',
    'attackingPlayersNorm',
    'defendingPlayersNorm',
    'ballsNorm',
    'threat_score',
    'threat_score_impact'
]

In [11]:
df_threat_features = (
    df_threat.select(*threat_cols)
    .join(
        df_features,
        on=["competitionId", "season", "gameId", "eventId"],
        how='inner'
    )
    .filter(~F.col("is_tracking_inconsistent"))  # remove eventos com tracking inconsistente
)

# flag: primeiro evento da posse
w_pos = Window.partitionBy("competitionId", "season", "gameId").orderBy("startGameClock")

df_threat_features = df_threat_features.withColumn(
    "first_cycle_event",
    F.lag("possession_id", 1).over(w_pos).isNull() |
    (F.lag("possession_id", 1).over(w_pos) != F.col("possession_id"))
)

# time que defende (oposto de quem tem a posse pela flag homeTeam): é o que as features descrevem
df_threat_features = df_threat_features.withColumn(
    "defendingTeamName",
    F.when(F.col("homeTeam"), F.col("opponentTeamName")).otherwise(F.col("homeTeamName"))
)

# posição x da bola (ballsNorm[0].x), usada na estratificação por zona
df_threat_features = df_threat_features.withColumn(
    "ball_x",
    F.get("ballsNorm", 0)["x"]
)

df_threat_features.show()

+-------------+---------+------+--------------------+----------+------+--------------+-----------------------+--------+----------------+------------+--------------------+-----------------------+-----------------------+------------------+-----------------------+------------------------+--------------+---------------+--------------+----------------+-------------+---------+---------+--------------------+--------------------+--------------------+------------+-------------------+------------------------+------------+-------------+-----------+----------+-------------+------------------+--------------------------+-------------------------+------------+------------+------------+-----------------------+-----------------------+-----------------+-----------------+------+
|competitionId|   season|gameId|             eventId|      date|period|startGameClock|startFormattedGameClock|homeTeam|flipped_homeTeam|   eventType|eventTypeDescription|eventSubTypeDescription|eventOutcomeDescription|   eventPla

## 3. Ciclos: goleiro + 11 atrás da bola

### 3.1 Filtro dos ciclos

Mantém ciclos cujo primeiro evento é do goleiro com 11 defensores em campo (o join traz todos os eventos desses ciclos) e mostra quanto isso representa do total.

In [12]:
df_cycle_start_gk_keys = (
    df_threat_features
    .filter(
        (F.col('first_cycle_event')) &
        (F.col('eventPlayerPositionType') == 'GK') &
        (F.col('defenders') == 11)
    )
    .select('competitionId', 'season', 'gameId', 'possession_id')
)

df_cycle_start_gk = (
    df_threat_features
    .join(
        F.broadcast(df_cycle_start_gk_keys),
        on=['competitionId', 'season', 'gameId', 'possession_id'],
        how='inner'
    )
)
df_cycle_start_gk = df_cycle_start_gk.cache()
df_cycle_start_gk.show(5)

+-------------+---------+------+-------------+--------------------+----------+------+--------------+-----------------------+--------+----------------+---------+--------------------+-----------------------+-----------------------+----------------+-----------------------+------------------------+-------------+---------------+------------+----------------+---------+---------+--------------------+--------------------+--------------------+------------+-------------------+------------------------+------------+-------------+-----------+----------+-------------+------------------+--------------------------+-------------------------+------------+------------+------------+-----------------------+-----------------------+-----------------+-----------------+------+
|competitionId|   season|gameId|possession_id|             eventId|      date|period|startGameClock|startFormattedGameClock|homeTeam|flipped_homeTeam|eventType|eventTypeDescription|eventSubTypeDescription|eventOutcomeDescription| eventPl

In [13]:
qtd_eventos = df_cycle_start_gk.count()
qtd_eventos_total = df_threat_features.count()

print(f'Quantidade de eventos: {qtd_eventos}')
print(f'Quantidade de eventos total na temporada: {qtd_eventos_total}')
print(f'Quantidade de eventos em relação ao total: {qtd_eventos / qtd_eventos_total:.2%}')

Quantidade de eventos: 26290
Quantidade de eventos total na temporada: 430937
Quantidade de eventos em relação ao total: 6.10%


In [14]:
df_possession_agg = (
    df_cycle_start_gk
    .groupBy('competitionId', 'season', 'gameId', 'possession_id')
    .agg(F.countDistinct(F.col('eventId')).alias('qtd_eventos')
    )
    .sort('qtd_eventos', ascending=False)
)

qtd_ciclos = df_possession_agg.count()
qtd_ciclos_total = df_threat_features.select('competitionId', 'season', 'gameId', 'possession_id').distinct().count()

print(f'Quantidade de ciclos: {qtd_ciclos}')
print(f'Quantidade total de ciclos na temporada: {qtd_ciclos_total}')
print(f'Quantidade de ciclos em relação ao total: {qtd_ciclos / qtd_ciclos_total:.2%}')

#df_possession_agg.show()

Quantidade de ciclos: 5570
Quantidade total de ciclos na temporada: 99034
Quantidade de ciclos em relação ao total: 5.62%


### 3.2 Inspeção de um jogo

Eventos e features dos ciclos filtrados de uma partida.

In [15]:
(
    df_cycle_start_gk
    .filter(F.col('gameId') == 4807)
    .select(
        'possession_id', 'period', 'startFormattedGameClock', 'eventTypeDescription', 'eventOutcomeDescription',
        'eventPlayerPositionType', 'eventPlayerPositionGroup', 'threat_score', 'is_tracking_inconsistent',
        'surface_area', 'stretch_index', 'team_length', 'team_width', 'defense_width',
        'height_goal_player', 'height_goal_team_centroide', 'height_goal_def_centroide',
        'def_mid_dist', 'def_atk_dist', 'atk_mid_dist',
        'numeric_superiority_10m', 'numeric_superiority_20m', 'ball_x',
    )
).show(3000, truncate=True)

+-------------+------+-----------------------+--------------------+-----------------------+-----------------------+------------------------+------------+------------------------+------------+-------------+-----------+----------+-------------+------------------+--------------------------+-------------------------+------------+------------+------------+-----------------------+-----------------------+------+
|possession_id|period|startFormattedGameClock|eventTypeDescription|eventOutcomeDescription|eventPlayerPositionType|eventPlayerPositionGroup|threat_score|is_tracking_inconsistent|surface_area|stretch_index|team_length|team_width|defense_width|height_goal_player|height_goal_team_centroide|height_goal_def_centroide|def_mid_dist|def_atk_dist|atk_mid_dist|numeric_superiority_10m|numeric_superiority_20m|ball_x|
+-------------+------+-----------------------+--------------------+-----------------------+-----------------------+------------------------+------------+------------------------+----

### 3.3 Tamanho dos ciclos

Distribuição da quantidade de eventos por ciclo.

In [16]:
df_possession_agg_pd = df_possession_agg.toPandas()

print(df_possession_agg_pd['qtd_eventos'].describe())

fig = go.Figure()

#cores = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

fig.add_trace(
    go.Box(
        y=df_possession_agg_pd['qtd_eventos'],
        name='',
        marker_color='#1f77b4',
        text=df_possession_agg_pd['possession_id'],
        hovertemplate='%{text}<br>Quantidade de eventos: %{y}<extra></extra>',
        #legendgroup=freq,
        #showlegend=True
    )
)

fig.update_layout(
    title=f'Ciclos de posse das partidas',
    height=700,
    width=600
)

fig.update_yaxes(title_text='Quantidade de eventos')
fig.show()

count    5570.000000
mean        4.717415
std         5.383034
min         1.000000
25%         1.000000
50%         3.000000
75%         6.000000
max        55.000000
Name: qtd_eventos, dtype: float64


### 3.4 Ciclos que continuam após o pico

% de ciclos em que o pico de ameaça não é o último evento (o time mantém a posse depois do pico), no total, por partida e por time.

In [17]:
# posição (rank) do evento de pico (max_by) vs total de eventos do ciclo
w_cycle_order = Window.partitionBy('competitionId', 'season', 'gameId', 'possession_id').orderBy('startGameClock')

df_cycle_ranked = df_cycle_start_gk.withColumn('event_rank', F.row_number().over(w_cycle_order))


df_peak_position = (
    df_cycle_ranked
    .groupBy('competitionId', 'season', 'gameId', 'possession_id')
    .agg(
        F.max_by(F.col('event_rank'), F.col('threat_score')).alias('peak_event_rank'),
        F.count('*').alias('total_events'),
    )
    .withColumn('continues_after_peak', F.col('peak_event_rank') < F.col('total_events'))
)

df_peak_position_pd = df_peak_position.toPandas()

qtd_continua = int(df_peak_position_pd['continues_after_peak'].sum())
qtd_total = len(df_peak_position_pd)
print(f'Ciclos que continuam com a posse depois do pico de ameaca: {qtd_continua} / {qtd_total} ({qtd_continua / qtd_total:.2%})')

# por partida (peso igual pra cada jogo)
df_continua_por_jogo = (
    df_peak_position_pd
    .groupby(['competitionId', 'season', 'gameId'])['continues_after_peak']
    .agg(qtd_ciclos='count', qtd_continua='sum')
)
df_continua_por_jogo['pct_continua'] = (df_continua_por_jogo['qtd_continua'] / df_continua_por_jogo['qtd_ciclos']).round(3)

media_pct_continua_temporada = df_continua_por_jogo['pct_continua'].mean()
print(f'Media do % de ciclos que continuam depois do pico, por partida: {media_pct_continua_temporada:.2%}')

#df_continua_por_jogo

Ciclos que continuam com a posse depois do pico de ameaca: 1537 / 5570 (27.59%)
Media do % de ciclos que continuam depois do pico, por partida: 27.73%


In [18]:
# por time (defendingTeamName)
df_cycle_team_keys = (
    df_cycle_start_gk
    .select('competitionId', 'season', 'gameId', 'possession_id', 'defendingTeamName')
    .distinct()
    .toPandas()
)

df_continua_por_time = (
    df_peak_position_pd
    .merge(df_cycle_team_keys, on=['competitionId', 'season', 'gameId', 'possession_id'], how='left')
    .groupby('defendingTeamName')['continues_after_peak']
    .agg(qtd_ciclos='count', qtd_continua='sum')
)
df_continua_por_time['pct_continua'] = (df_continua_por_time['qtd_continua'] / df_continua_por_time['qtd_ciclos']).round(3)

media_pct_continua_por_time = df_continua_por_time['pct_continua'].mean()
print(f'Media do % de ciclos que continuam depois do pico, por time: {media_pct_continua_por_time:.2%}')

df_continua_por_time.sort_values('pct_continua', ascending=True)

Media do % de ciclos que continuam depois do pico, por time: 28.13%


,qtd_ciclos,qtd_continua,pct_continua
defendingTeamName,,,
Liverpool,377,81,0.215
Brighton & Hove Albion,348,77,0.221
Manchester City,338,78,0.231
Chelsea,317,75,0.237
Manchester United,324,77,0.238
Brentford,239,61,0.255
Tottenham Hotspur,279,72,0.258
Newcastle United,249,67,0.269
Arsenal,361,100,0.277


## 4. Correlação features x ameaça

### 4.1 Agregação por ciclo

Features agregadas por ciclo (todas as agregações) e ameaça (avg/sum/max).

In [19]:
features = [
    'surface_area',
    'stretch_index',
    'team_length',
    'team_width',
    'defense_width',
    'height_goal_player',
    'height_goal_team_centroide',
    'height_goal_def_centroide',
    'def_mid_dist',
    'def_atk_dist',
    'atk_mid_dist',
    'numeric_superiority_10m',
    'numeric_superiority_20m',
]
target = 'threat_score'
group_cols = ['competitionId', 'season', 'gameId', 'possession_id']

df_features_target_agg = build_feature_target_stats_df(
    df=df_cycle_start_gk,
    features=features,
    target=target,
    group_cols=group_cols,
)

df_features_target_agg_pd = df_features_target_agg.toPandas()

# for feature in features:
#     plot_correlation_heatmap(df_features_target_agg_pd, feature, target, target_aggs=['max'])

### 4.2 Agregações mais relevantes

Para cada feature, testa todas as agregações contra `max_threat_score` e guarda as 2 de maior correlação absoluta com p < 0.05 (`AGG_FEATURES`). O resultado é salvo em `data/relevant_features.json`, lido pelo notebook do modelo.

In [20]:
# testa todas as agregações de cada feature contra max_threat_score; mantém as 2 de maior |correlação| com p < 0.05
target_col = f'max_{target}'
agg_names = list(AGG_FUNCS_FEATURES.keys())

AGG_FEATURES = {}

for feature in features:
    results = []
    for prefix in agg_names:
        feature_col = f'{prefix}_{feature}'
        paired = df_features_target_agg_pd[[feature_col, target_col]].dropna()
        if len(paired) < 3:
            continue
        r, pvalue = stats.spearmanr(paired[feature_col], paired[target_col])
        if pvalue < 0.05:
            results.append((prefix, r, pvalue))

    results.sort(key=lambda item: abs(item[1]), reverse=True)
    AGG_FEATURES[feature] = [
        {prefix: round(r, 3), 'p-value': round(pvalue, 3)}
        for prefix, r, pvalue in results[:2]
    ]

for feature, aggs in AGG_FEATURES.items():
    parts = []
    for d in aggs:
        prefix = next(key for key in d if key != 'p-value')
        parts.append(f"{prefix.capitalize()}: {d[prefix]} (p-valor: {d['p-value']})")
    print(f'Feature: {feature}')
    print(f"Agregações: {', '.join(parts)}")
    print()

Feature: surface_area
Agregações: Min: -0.362 (p-valor: 0.0), Max: 0.193 (p-valor: 0.0)

Feature: stretch_index
Agregações: Min: -0.363 (p-valor: 0.0), Max: 0.175 (p-valor: 0.0)

Feature: team_length
Agregações: Min: -0.459 (p-valor: 0.0), Median: -0.254 (p-valor: 0.0)

Feature: team_width
Agregações: Max: 0.407 (p-valor: 0.0), Min: -0.242 (p-valor: 0.0)

Feature: defense_width
Agregações: Max: 0.356 (p-valor: 0.0), Min: -0.263 (p-valor: 0.0)

Feature: height_goal_player
Agregações: Min: -0.732 (p-valor: 0.0), Avg: -0.587 (p-valor: 0.0)

Feature: height_goal_team_centroide
Agregações: Min: -0.704 (p-valor: 0.0), Avg: -0.542 (p-valor: 0.0)

Feature: height_goal_def_centroide
Agregações: Min: -0.713 (p-valor: 0.0), Avg: -0.557 (p-valor: 0.0)

Feature: def_mid_dist
Agregações: Min: -0.344 (p-valor: 0.0), Max: 0.179 (p-valor: 0.0)

Feature: def_atk_dist
Agregações: Min: -0.363 (p-valor: 0.0), Max: 0.193 (p-valor: 0.0)

Feature: atk_mid_dist
Agregações: Min: -0.34 (p-valor: 0.0), Max: 0.261

In [21]:
# salva AGG_FEATURES em data/relevant_features.json (lido pelo notebook do modelo)
import json

relevant_features_path = data_folder_path / "relevant_features.json"

with open(relevant_features_path, "w", encoding="utf-8") as f:
    json.dump(AGG_FEATURES, f, indent=2, ensure_ascii=False, default=float)

print(f"Salvo em: {relevant_features_path}")

Salvo em: C:\Users\MatheusSantos\Documents\Projetos\Mestrado\defensive-performance-prediction\data\relevant_features.json


### 4.3 Heatmaps por time

A correlação se sustenta time a time? Um bloco por time (até `max_teams`), com um subplot por posição no ranking de agregações.

In [22]:
# mesma agregação, com o time (defendingTeamName) nas chaves de agrupamento
group_cols_by_team = ['competitionId', 'season', 'gameId', 'possession_id', 'defendingTeamName']

df_features_target_agg_by_team = build_feature_target_stats_df(
    df=df_cycle_start_gk,
    features=features,
    target=target,
    group_cols=group_cols_by_team,
)

df_features_target_agg_by_team_pd = df_features_target_agg_by_team.toPandas()

plot_correlation_heatmap_by_team(
    df_features_target_agg_by_team_pd, features, target, team_col='defendingTeamName', feature_agg=AGG_FEATURES, target_aggs='max', max_teams=3
)

## 5. Estratificação por posição de campo

Testa se a correlação se sustenta em diferentes zonas do campo ou é confundimento posicional: o pico de ameaça e a compactação defensiva tendem a aumentar juntos perto do gol.

### 5.1 Posição da bola no pico

Posição x da bola no evento de pico de ameaça de cada ciclo, dividida em faixas (`pd.cut`).

In [23]:
# ball_x do evento de pico do target no ciclo (max_by)

df_ball_pos_at_max = (
    df_cycle_start_gk
    .groupBy(*group_cols)
    .agg(F.max_by(F.col('ball_x'), F.col(target)).alias('ball_x_at_max'))
)

df_features_target_agg_pos = df_features_target_agg.join(df_ball_pos_at_max, on=group_cols, how='inner')

df_features_target_agg_pos_pd = df_features_target_agg_pos.toPandas()

# faixas criadas depois da agregação: dependem do resultado do max_by, então não podem ser chave do groupBy
NUM_POSITION_BINS = 4

df_features_target_agg_pos_pd['ball_x_at_max_bin'] = pd.cut(
    df_features_target_agg_pos_pd['ball_x_at_max'], bins=NUM_POSITION_BINS
)

### 5.2 Teste de composição

As faixas são zona real ou posse curta disfarçada? Compara a quantidade de eventos por ciclo em cada faixa.

In [24]:
# qtd_eventos por ciclo (df_possession_agg_pd), por faixa
df_composicao_por_faixa = (
    df_features_target_agg_pos_pd
    .merge(
        df_possession_agg_pd,
        on=['competitionId', 'season', 'gameId', 'possession_id'],
        how='left'
    )
    .groupby('ball_x_at_max_bin')['qtd_eventos']
    .agg(['mean', 'median', 'count'])
    .round(2)
)

df_composicao_por_faixa

,mean,median,count
ball_x_at_max_bin,,,
"(-52.603, -24.193]",1.66,1.0,2610
"(-24.193, 4.105]",5.16,4.0,1304
"(4.105, 32.403]",7.11,5.0,944
"(32.403, 60.7]",11.95,10.0,712


### 5.3 Correlação por faixa

Heatmap features x faixas, com a agregação mais relevante de cada feature.

In [25]:
plot_correlation_by_position_bin(
    df_features_target_agg_pos_pd, features, target,
    feature_agg=AGG_FEATURES, target_agg='max', bin_col='ball_x_at_max_bin'
)

### 5.4 Correlação por faixa e por time

Mesma estratificação, time a time.

In [26]:
# ball_x_at_max com o time nas chaves de agrupamento

df_ball_pos_at_max_by_team = (
    df_cycle_start_gk
    .groupBy(*group_cols_by_team)
    .agg(F.max_by(F.col('ball_x'), F.col(target)).alias('ball_x_at_max'))
)

df_features_target_agg_by_team_pos = df_features_target_agg_by_team.join(
    df_ball_pos_at_max_by_team, on=group_cols_by_team, how='inner'
)

df_features_target_agg_by_team_pos_pd = df_features_target_agg_by_team_pos.toPandas()

df_features_target_agg_by_team_pos_pd['ball_x_at_max_bin'] = pd.cut(
    df_features_target_agg_by_team_pos_pd['ball_x_at_max'], bins=NUM_POSITION_BINS
)

plot_correlation_by_position_bin_by_team(
    df_features_target_agg_by_team_pos_pd, features, target,
    team_col='defendingTeamName', feature_agg=AGG_FEATURES, target_agg='max', bin_col='ball_x_at_max_bin', max_teams=3
)

## 6. Ciclos mais longos

Repete as análises por faixa só com ciclos de pelo menos `MIN_QTD_EVENTOS` eventos, para reduzir o ruído de posses curtas.

### 6.1 Por faixa

In [27]:
# só ciclos com >= MIN_QTD_EVENTOS eventos (reaproveita os bins já calculados)
MIN_QTD_EVENTOS = 3

df_features_target_agg_pos_min_eventos_pd = (
    df_features_target_agg_pos_pd
    .merge(
        df_possession_agg_pd,
        on=['competitionId', 'season', 'gameId', 'possession_id'],
        how='left'
    )
    .query(f'qtd_eventos >= {MIN_QTD_EVENTOS}')
)

print(df_features_target_agg_pos_min_eventos_pd['ball_x_at_max_bin'].value_counts().sort_index())

plot_correlation_by_position_bin(
    df_features_target_agg_pos_min_eventos_pd, features, target,
    feature_agg=AGG_FEATURES, target_agg='max', bin_col='ball_x_at_max_bin'
)

ball_x_at_max_bin
(-52.603, -24.193]    443
(-24.193, 4.105]      997
(4.105, 32.403]       821
(32.403, 60.7]        696
Name: count, dtype: int64


### 6.2 Por faixa e por time

In [28]:
# só ciclos com >= MIN_QTD_EVENTOS eventos
df_features_target_agg_by_team_pos_min_eventos_pd = (
    df_features_target_agg_by_team_pos_pd
    .merge(df_possession_agg_pd, on=['competitionId', 'season', 'gameId', 'possession_id'], how='left')
    .query(f'qtd_eventos >= {MIN_QTD_EVENTOS}')
)

plot_correlation_by_position_bin_by_team(
    df_features_target_agg_by_team_pos_min_eventos_pd, features, target,
    team_col='defendingTeamName', feature_agg=AGG_FEATURES, target_agg='max', bin_col='ball_x_at_max_bin', max_teams=3
)

## 7. Conclusões e próximos passos

- OBS: A intenção é trazer uma v0 pra qualificação, mas é possivel melhorar o modelo pensando em estratégias pós qualificação para dissertação ou artigos

- OBS2: A analise olhando ciclos começando no goleiro e os 11 atrás foi melhor que a começando com apenas 11 atrás, pois sugere-se que dá pra ver melhor as estratificações em campo, enquanto que sem começar no goleiro pode começar em qualquer parte do campo e fica pior.

Os times mostraram ter variações muito signficativas, mas o modelo deve ser um formato default primeiro como v0 e considerando todos dentro do mesmo bolo.

Legal que cortar eventos curtos reduziu ruido no primeiro quadrante à esquerda e mostrou caracteristicas de sucesso na performance a respeito de pressão defensiva alta entre os times. O problema é que variando o limiar minimo de evento pro ciclo ser valido reduz a amosra que já é pequena. Vale testar mudando esse limiar caso necessário, mas a v0 ser sem esse corte ou começando com u mlimiar baixo de 1-2 eventos.

### Próximos passos (notas)

- sugestão: correlacionar o max_threat_score com as minimas x qtd de gols sofridos por jogo ou xG sofrido
- levar essas melhores agregações pra uma v0 do modelo (dps daria pra explorar todas as variações de uma vez pro modelo e ver pela explicabilidade qual foi melhor e se bate com a eda)
- ver resultados/performance e pensar no que da pra melhorar (features, algoritmos, etc)